# ATOMICAScore: ranking interface residues by how much the ligand depends on them

ATOMICAScore masks one interface residue at a time and measures how far that moves the pretrained model's
representation of the **ligand**. For block $i$ of an interaction graph $G$, build $G \setminus i$ by replacing
that block with the mask block and its atoms with a single mask atom, then

$$a_i = \cos\big(\mathbf{r}(G),\ \mathbf{r}(G \setminus i)\big)$$

A **low** $a_i$ means masking the residue changed the readout a lot, so that residue matters more.

The readout $\mathbf{r}$ is the component-normalized mean of $\mathbf{z}^{\mathrm{block}}$ over the ligand's
blocks. It comes from `atomica.representations`, the one place a representation is defined.

Needs the `atomica` environment and the pretrained checkpoint, which the notebook downloads if missing.
A GPU is faster but not required.

## 1. Locate the repository

In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)


def find_repo_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / "pyproject.toml").exists() and (parent / "src" / "atomica").exists():
            return parent
    raise RuntimeError(f"Could not locate the ATOMICA repository root from {start}")


REPO_ROOT = find_repo_root(Path(os.getcwd()).resolve())
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
print(f"repository root: {REPO_ROOT}")

repository root: /n/holylabs/mzitnik_lab/Users/afang/ATOMICA-public


## 2. Download the pretrained checkpoint

Skip if you already have it. Needs the Hugging Face CLI (`pip install -U "huggingface_hub[cli]"`).

In [2]:
ckpt_dir = REPO_ROOT / "checkpoints" / "ATOMICA_checkpoints" / "pretrain"
config_path = ckpt_dir / "pretrain_model_config.json"
weights_path = ckpt_dir / "pretrain_model_weights.pt"

if not (config_path.exists() and weights_path.exists()):
    import subprocess

    subprocess.run(
        ["hf", "download", "ada-f/ATOMICA", "--repo-type", "model",
         "--local-dir", str(REPO_ROOT / "checkpoints"),
         "--include", "ATOMICA_checkpoints/pretrain/**"],
        check=True,
    )

print(f"config:  {config_path}")
print(f"weights: {weights_path}")

config:  /n/holylabs/mzitnik_lab/Users/afang/ATOMICA-public/checkpoints/ATOMICA_checkpoints/pretrain/pretrain_model_config.json
weights: /n/holylabs/mzitnik_lab/Users/afang/ATOMICA-public/checkpoints/ATOMICA_checkpoints/pretrain/pretrain_model_weights.pt


## 3. Process the example structures

`process_pdbs` turns each structure into the block-level interaction graph the model consumes, and records
`block_to_pdb_indexes`, which maps every block back to its chain and residue number. We use `6llw_A_A_UDP`,
a glycosyltransferase bound to UDP.

In [3]:
import pandas as pd

processed_data_path = REPO_ROOT / "data" / "example" / "example_processed_data.parquet"
input_csv = REPO_ROOT / "data" / "example" / "example_inputs.csv"

if not processed_data_path.exists():
    import subprocess

    subprocess.run(
        ["python", "-m", "atomica.data.process_pdbs",
         "--data_index_file", str(input_csv), "--out_path", str(processed_data_path),
         "--interface_dist_th", "8.0", "--fragmentation_method", "PS_300"],
        check=True, cwd=REPO_ROOT,
    )

EXAMPLE_ID = "6llw_A_A_UDP"
pd.read_parquet(processed_data_path)[["id"]]

,id
0,6llw_A_A_UDP
1,3i5x_A_B
2,5kl2_A_BC
3,6d1u_A_D
4,2uxq_A_B
5,6hrg_A_A_ZN
6,4yaz_A_A_4BW


## 4. Load the model

Every name `atomica.representations` exposes corresponds to a symbol in the paper, so a name in a script, in a
saved file and in the paper mean the same object.

In [4]:
import torch

from atomica import representations as R
from atomica.data.dataset import PDBDataset
from atomica.data.pdb_utils import VOCAB
from atomica.models.prediction_model import PredictionModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model = PredictionModel.load_from_config_and_weights(str(config_path), str(weights_path))
model = model.to(device).eval()

print(R.describe(model))

Pretrained model params: hidden_size=32,
               edge_size=32, k_neighbors=8, 
               n_layers=4, bottom_global_message_passing=False,
               global_message_passing=True, 
               fragmentation_method=PS_300


name         paper          level      family  width       
-----------  -------------  ---------  ------  ------------
h_atom       h_a^atom       atom       h       32          
h_block      h_b^block      block      h       32          
h_graph      h_i^graph      graph      h       32          
h_interface  h_A^interface  interface  h       32          
z_atom       z_a^atom       atom       z       608         
z_block      z_b^block      block      z       1792        
z_graph      z_i^graph      graph      z       5376 or 1792
z_interface  z_i^interface  interface  z       5376 or 1792

global_message_passing=True: h_graph includes the global block node.


## 5. The readout

`z_block` is three parts concatenated. Component normalization scales each to unit length so the cosine weighs
them equally instead of in proportion to their magnitude. The second cell shows why that matters.

In [5]:
from atomica.interaction_profiler.interact_score import BATCH_SIZE, mask_block

dataset = PDBDataset(str(processed_data_path))
index = dataset.indexes.index(EXAMPLE_ID)
data = dataset[index]
block_to_pdb = dataset.data[index]["block_to_pdb_indexes"]


def ligand_pooled(graph):
    """Mean of z_block over the ligand blocks, before normalization.

    The batch is `graph` followed by copies of the intact complex, which is the company the score keeps.
    The cross-attention pads every block out to the largest block in the batch, so this holds the pad
    width fixed for both sides of the comparison below.
    """
    batch = PDBDataset.collate_fn([graph] + [data] * (BATCH_SIZE - 1))
    batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    with torch.no_grad():
        per_block = R.get(model, batch, "z_block")
    first = batch["B"].new_zeros(len(batch["B"]), dtype=torch.bool)
    first[:int(batch["lengths"][0])] = True
    keep = first & (batch["segment_ids"] == 1) & (batch["B"] != model.global_block_id)
    return per_block[keep].mean(0)


unnormalized = ligand_pooled(data)
print("the three parts of z_block:", model.invariant_component_dims())
print("\nL2 norm of each part before normalization:")
offset = 0
for name, width in model.invariant_component_dims().items():
    print(f"  {name:8s} width {width:5d}   norm {unnormalized[offset:offset + width].norm():.3f}")
    offset += width

the three parts of z_block: {'h_block': 32, 'gram': 544, 'atom': 1216}

L2 norm of each part before normalization:
  h_block  width    32   norm 0.981
  gram     width   544   norm 2.569
  atom     width  1216   norm 11.966


An un-normalized cosine is the sum of the three parts' inner products. Splitting it shows how lopsided the
weighting is: masking a residue moves all three parts, but almost the whole cosine is the atom part.

In [6]:
trp340 = next(b for b, tag in block_to_pdb.items() if tag == "A_340")
masked_top = ligand_pooled(mask_block(data, trp340))

contributions, offset = {}, 0
for name, width in model.invariant_component_dims().items():
    contributions[name] = float((unnormalized[offset:offset + width]
                                 * masked_top[offset:offset + width]).sum())
    offset += width

total = sum(contributions.values())
print("share of the un-normalized cosine contributed by each part:")
for name, value in contributions.items():
    print(f"  {name:8s} {100 * value / total:6.2f}%")

share of the un-normalized cosine contributed by each part:
  h_block    0.63%
  gram       4.33%
  atom      95.04%


## 6. Compute ATOMICAScore

The intact complex sits in slot 0 of every forward pass and each pass is padded to a fixed `BATCH_SIZE`, which
the result records. Both sides of every cosine then share the cross-attention pad width, which is what the
published implementation got by scoring one `[intact, masked]` pair at a time.

In [7]:
import numpy as np

from atomica.interaction_profiler.interact_score import (
    atomica_score, auroc, find_ligand_segment, precision_at_k, scorable_blocks)

print("ligand segment (inferred):", find_ligand_segment(data))

result = atomica_score(model, data, device=device)
print(f"scored {len(result)} amino-acid residue blocks at batch size {result.batch_size}")
print(f"readout: {result.readout} pooled by {result.pooling}")
print(f"score range: {result.score.min():.5f} to {result.score.max():.5f}")

wider = atomica_score(model, data, device=device, batch_size=16)
print(f"\nat batch size 16: max|delta| = {np.abs(result.score - wider.score).max():.3e}, "
      f"ranking unchanged: {result.ranking() == wider.ranking()}")

ligand segment (inferred): 1


scored 29 amino-acid residue blocks at batch size 8
readout: z_interface pooled by mean_component_normalized
score range: 0.99744 to 0.99992



at batch size 16: max|delta| = 1.192e-07, ranking unchanged: True


## 7. Rank the residues

`block_to_pdb_indexes` maps each block back to its chain and residue number in the deposited structure.

In [8]:
ranked = pd.DataFrame({
    "block_idx": result.block_idx,
    "pdb_residue": [block_to_pdb.get(b) for b in result.block_idx],
    "residue_type": [VOCAB.idx_to_symbol(int(data["B"][b])) for b in result.block_idx],
    "atomica_score": result.score,
}).sort_values("atomica_score").reset_index(drop=True)
ranked.index += 1
ranked.head(12)

,block_idx,pdb_residue,residue_type,atomica_score
1,15,A_340,W,0.997444
2,9,A_279,N,0.997938
3,18,A_343,Q,0.998306
4,25,A_362,N,0.998310
5,8,A_278,G,0.998322
6,10,A_280,R,0.998514
7,21,A_358,H,0.999080
8,26,A_363,S,0.999090
9,16,A_341,V,0.999250
10,29,A_366,E,0.999281


## 8. Evaluate against PLIP annotations

A residue is positive when [PLIP](https://plip-tool.biotec.tu-dresden.de) annotates it in a non-covalent
interaction with the ligand: hydrogen bonds, hydrophobic interactions, pi-stacking, metal complexes or halogen
bonds. `make_plip_labels.py` regenerates the annotations if you install PLIP.

precision@10 is a fraction of ten.

In [9]:
labels = pd.read_csv(REPO_ROOT / "data" / "example" / "example_plip_labels.csv")
annotated = set(labels.loc[labels.complex_id == EXAMPLE_ID, "pdb_residue"])

print(labels[labels.complex_id == EXAMPLE_ID][["pdb_residue", "restype", "interaction_type"]]
      .to_string(index=False))

pdb_residue restype interaction_type
      A_279     ASN   Hydrogen Bonds
      A_280     ARG   Hydrogen Bonds
      A_340     TRP      pi-Stacking
      A_341     VAL   Hydrogen Bonds
      A_343     GLN   Hydrogen Bonds
      A_362     ASN   Hydrogen Bonds
      A_363     SER   Hydrogen Bonds
      A_366     GLU   Hydrogen Bonds


In [10]:
y = np.array([block_to_pdb.get(b) in annotated for b in result.block_idx])

print(f"scored residues      : {len(y)}")
print(f"annotated among them : {int(y.sum())}")
print(f"precision@10         : {precision_at_k(result.importance, y, k=10):.3f}")
print(f"AUROC                : {auroc(result.importance, y):.3f}")

ranked["annotated"] = [block_to_pdb.get(b) in annotated for b in ranked["block_idx"]]
ranked.head(12)

scored residues      : 29
annotated among them : 8
precision@10         : 0.800
AUROC                : 0.958


,block_idx,pdb_residue,residue_type,atomica_score,annotated
1,15,A_340,W,0.997444,True
2,9,A_279,N,0.997938,True
3,18,A_343,Q,0.998306,True
4,25,A_362,N,0.998310,True
5,8,A_278,G,0.998322,False
6,10,A_280,R,0.998514,True
7,21,A_358,H,0.999080,False
8,26,A_363,S,0.999090,True
9,16,A_341,V,0.999250,True
10,29,A_366,E,0.999281,True


## 9. Scope

ATOMICAScore is defined over amino-acid residue blocks, because a protein block is one residue while a
small-molecule block is a chemical fragment. `4yaz_A_A_4BW` is a riboswitch, so its receptor is RNA and the
result is empty rather than a ranking that would not mean what it appears to mean.

`6hrg_A_A_ZN` is a zinc site with only 9 interface residues. The score puts all four metal-coordinating
residues in the top six, but precision@10 is not informative below ten residues, which is why the published
evaluation keeps only complexes with at least 12.

In [11]:
riboswitch = dataset[dataset.indexes.index("4yaz_A_A_4BW")]
print("amino-acid blocks in 4yaz_A_A_4BW:", scorable_blocks(riboswitch, ligand_segment=1))
print("residues scored:", len(atomica_score(model, riboswitch, ligand_segment=1, device=device)))

amino-acid blocks in 4yaz_A_A_4BW: []
residues scored: 0


In [12]:
zinc_index = dataset.indexes.index("6hrg_A_A_ZN")
zinc = dataset[zinc_index]
zinc_map = dataset.data[zinc_index]["block_to_pdb_indexes"]

zinc_result = atomica_score(model, zinc, device=device)
zinc_annotated = set(labels.loc[labels.complex_id == "6hrg_A_A_ZN", "pdb_residue"])
zinc_y = np.array([zinc_map.get(b) in zinc_annotated for b in zinc_result.block_idx])

print(f"6hrg_A_A_ZN: {len(zinc_result)} scored residues, {int(zinc_y.sum())} annotated")
print(f"AUROC: {auroc(zinc_result.importance, zinc_y):.3f}")

pd.DataFrame({
    "pdb_residue": [zinc_map.get(b) for b in zinc_result.block_idx],
    "residue_type": [VOCAB.idx_to_symbol(int(zinc["B"][b])) for b in zinc_result.block_idx],
    "atomica_score": zinc_result.score,
    "coordinates_zinc": zinc_y,
}).sort_values("atomica_score").reset_index(drop=True)

6hrg_A_A_ZN: 9 scored residues, 4 annotated
AUROC: 0.850


,pdb_residue,residue_type,atomica_score,coordinates_zinc
0,A_59,H,0.889679,True
1,A_58,D,0.905621,True
2,A_145,T,0.936551,False
3,A_194,H,0.946023,True
4,A_56,H,0.948345,False
5,A_144,D,0.952101,True
6,A_57,E,0.960935,False
7,A_12,S,0.981504,False
8,A_172,T,0.983351,False


## 10. Scoring your own structures

Process your structures, then score them:

```bash
python -m atomica.data.process_pdbs     --data_index_file my_inputs.csv --out_path my_processed.parquet     --interface_dist_th 8.0 --fragmentation_method PS_300

python -m atomica.interaction_profiler.interact_score     --data_path my_processed.parquet --output_path my_scores.jsonl     --model_config  checkpoints/ATOMICA_checkpoints/pretrain/pretrain_model_config.json     --model_weights checkpoints/ATOMICA_checkpoints/pretrain/pretrain_model_weights.pt
```

Each output line holds the complex id, the scored block indices, the scores, the ligand segment, the batch
size and the readout. Complexes with no amino-acid residue block, or an ambiguous ligand side, are skipped
and counted.